Step 0: Install Dependencies


In [1]:
# [Cell 0] Install Dependencies
!pip install -q transformers datasets peft accelerate huggingface_hub gradio

In [2]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.8Gi       7.1Gi       2.0Mi       3.8Gi        10Gi
Swap:             0B          0B          0B


In [3]:
!nvidia-smi

Mon May 11 19:21:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Step 1: Environment Setup and Authentication
Before diving into the code, it is important to note that a Hugging Face token is not strictly required to download public models like Qwen. However, if you do not provide one, the Hugging Face library will generate a warning message. Authenticating is a good practice as it suppresses these warnings and is mandatory if you ever decide to use restricted or private models (like Llama 3).

To set this up, add your Hugging Face Access Token to the Secrets tab in Google Colab (the key icon on the left sidebar) and name it HF_TOKEN. Enable notebook access for this secret.

In [3]:
#[Cell 1] Environment Setup & Authentication
from google.colab import userdata
from huggingface_hub import login
import warnings

# Suppress the specific warning about missing tokens
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub.utils._auth")

try:
    # Attempt to fetch the token from Colab's secure storage
    hf_token = userdata.get('HF_TOKEN')

    # Log into the Hugging Face Hub using the retrieved token
    login(hf_token)
    print("Authentication successful.")
except userdata.SecretNotFoundError:
    # If the token isn't found, smoothly continue without crashing
    print("Notice: HF_TOKEN not found in Colab secrets. Proceeding without authentication.")

Notice: HF_TOKEN not found in Colab secrets. Proceeding without authentication.


Step 2: Load the Starting Model and Tokenizer
What is a Tokenizer? Neural networks cannot read text. The tokenizer translates text strings into numerical IDs (tokens) that the model can process mathematically, and later converts the model's numerical output back into human-readable text.

You can explore how tokenization works in practice using this interactive demo: https://platform.openai.com/tokenizer

In [5]:
#[Cell 2] Load Model and Tokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# 1. Load and configure the Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# 2. Load and configure the Neural Network Model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="cuda",
    torch_dtype=torch.float16
)

model.config.use_cache = False

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Step 2b: Baseline Inference (Before Fine-Tuning)
Before training, ask the model a target question. Because the starting model has never seen MediCore.json, it will likely give a generic or hallucinated answer about MediCore Hospital. This establishes a baseline to prove that fine-tuning changed the model's behavior.

In [6]:
# [Cell 2b] Baseline Inference (Before Fine-Tuning)
messages = [{"role": "user", "content": "Who leads the neurology department at MediCore Hospital?"}]

# Format the prompt
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)
# print("Token IDs:\n", inputs["input_ids"][0][:20]) # Shows the first ~20 token IDs (numbers)

# Generate an answer using the untrained starting model
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)

# Extract and print only the generated response
generated_ids = output[0][inputs.input_ids.shape[1]:]
print("STARTING MODEL ANSWER:\n", tokenizer.decode(generated_ids, skip_special_tokens=True))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


STARTING MODEL ANSWER:
 I'm sorry, but I can't answer this question. This might be a sensitive and personal matter that should be discussed with a healthcare professional or contacted directly through their official channels. As an AI language model, I don't have access to private information


Step 3: Load and Preprocess the Dataset
To get the dataset, you have two options: You can either manually upload your MediCore.json file to the Colab file system (using the folder icon on the left sidebar), or let the code automatically download it for you. This step fetches the data and maps it into the standardized ChatML template.

-nc (no-clobber) ensures that if you manually uploaded your own version of MediCore.json, the command will not overwrite it.
-q (quiet) prevents it from printing a messy download progress bar in the Colab output.


In [7]:
# Automatically download the dataset if it hasn't been uploaded manually
!wget -nc -q https://github.com/ML-Course-2026/session6/raw/refs/heads/main/material/datasets/MediCore.json

In [8]:
#[Cell 3] Dataset Loading and Preprocessing
from datasets import load_dataset

raw_data = load_dataset("json", data_files="MediCore.json")

def preprocess(sample):
    messages = [
        {"role": "user", "content": sample['prompt']},
        {"role": "assistant", "content": sample['completion']}
    ]

    # Automatically applies <|im_start|> and <|im_end|> ChatML tags
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    tokenized = tokenizer(
        text,
        truncation=True,
        #max_length=256,
        padding=False
    )
    # Explicitly create labels for loss calculation
    #tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

data = raw_data.map(
    preprocess,
    remove_columns=raw_data["train"].column_names
)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/486 [00:00<?, ? examples/s]

Step 4: Configure PEFT and LoRA Adapters
What is PEFT? PEFT (Parameter-Efficient Fine-Tuning) is both an umbrella concept and an official Hugging Face library. Instead of updating every single parameter in a massive neural network (which requires supercomputers), PEFT methods freeze the original model and only train a tiny fraction of new parameters. The peft library handles all the complex PyTorch code required to do this automatically.

What is LoRA? Large Language Models possess billions of parameters. Updating all of them simultaneously (Full Fine-Tuning) requires immense computing power and VRAM. Low-Rank Adaptation (LoRA) is a technique that freezes the original model weights and injects small, trainable "adapter" matrices into specific layers (like the attention mechanism's q_proj and v_proj). You achieve ~90% of the quality of full fine-tuning while training only ~1% of the parameters. LoRA (Low-Rank Adaptation) is the most popular specific technique inside the PEFT library.

In [9]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, message="TypedStorage is deprecated")

# Upgrade torchao to a compatible version
!pip install --upgrade torchao

# [Cell 4] LoRA Configuration
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none"
)

model = get_peft_model(model, lora_config)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 41.2 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


Step 5: Configure Training Arguments and Execute Training
Understanding the Step Count: If your output shows a particular number of training steps, that number comes from the size of the training split, the batch settings, gradient accumulation, and the number of epochs.

A useful simplified intuition is: Total optimizer steps ≈ (Training Split Size ÷ Effective Batch Size) × Epochs

In this lab, the exact count depends on all of the following:

10% of the dataset is reserved for validation,
per_device_train_batch_size=1,
gradient_accumulation_steps=2,
num_train_epochs=5.
So the exact step count will vary with your dataset size and with how the Trainer rounds partial batches.



In [10]:
#[Cell 5] Training Setup and Execution
from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer

# Let the collator handle padding + labels
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Split 10% of the data for validation
split = data["train"].train_test_split(test_size=0.1)
train_dataset = split["train"]
eval_dataset = split["test"]

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    learning_rate=2e-4,

    per_device_train_batch_size=3,      # ↓ reduce to avoid OOM
    gradient_accumulation_steps=4,      # keeps effective batch size

    fp16=True,                          # ↓ big memory saver

    logging_steps=5,
    eval_strategy="epoch",
    lr_scheduler_type="cosine",
    remove_unused_columns=False
)

# IMPORTANT: enable memory savings
model.gradient_checkpointing_enable()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator
)

trainer.train()

trainer.save_model("./my_qwen")
tokenizer.save_pretrained("./my_qwen")

Epoch,Training Loss,Validation Loss
1,0.596511,0.625236
2,0.468415,0.602607
3,0.345219,0.664722
4,0.229299,0.733282
5,0.192826,0.790560


('./my_qwen/tokenizer_config.json',
 './my_qwen/chat_template.jinja',
 './my_qwen/tokenizer.json')

Step 6: Load the Fine-Tuned Model
Once training is complete, the LoRA adapters must be loaded alongside the starting model. This cell simulates what you would do if you restarted your Colab notebook and wanted to load your saved work.

Note

If you have NOT restarted your runtime, skip this step. Your model is already in memory from Step 5.

In [11]:
# [Cell 6] Load Model for Testing
from peft import PeftModel, PeftConfig

path = "./my_qwen"
config = PeftConfig.from_pretrained(path)

# 1. Load the original starting-model checkpoint
base_model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    device_map="cuda",
    torch_dtype=torch.float16
)

# 2. Attach your tiny, fine-tuned adapter to the starting model
model = PeftModel.from_pretrained(base_model, path)

# 3. Re-enable caching for faster inference speeds
model.config.use_cache = True

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Step 7: Test the Fine-Tuned Model
This step tests the model using the proper ChatML format and greedy decoding to retrieve the exact factual data injected during training.



In [12]:
# [Cell 7] Inference Execution
messages =[ {"role": "user", "content": "Who leads the neurology department at MediCore Hospital?"} ]

# Format the text with ChatML tags and generation prompt
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Convert text to tensor numbers and move to GPU
inputs = tokenizer(text, return_tensors="pt").to(model.device)

# Generate the output
output = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)

# Strip out the input prompt so we only see the newly generated answer
generated_ids = output[0][inputs.input_ids.shape[1]:]
print("FINE-TUNED ANSWER:\n", tokenizer.decode(generated_ids, skip_special_tokens=True))

FINE-TUNED ANSWER:
 Dr. Elena Salonen leads the neurology department at MediCore Hospital.


Core project extension: structured JSON and Gradio
Note

The cell numbering intentionally keeps the original working notebook order. Optional merge-and-save remains as Cell 8 in Appendix A at the end, so the main JSON/Gradio extension continues with Cells 9 to 12 here.

Concept: Ensuring Structured Outputs (Markdown or JSON)
When integrating a language model into a user interface like Gradio, you often need the output to be strictly formatted.

Markdown is ideal if you want Gradio to render rich text (bolding, lists, tables).
JSON is ideal if you want Gradio (or another Python script) to programmatically parse the response into dictionaries and variables.
Language models are pattern matchers. To guarantee they output a specific format, you must combine System Prompts with Dataset Formatting.

Important

For the mini project, JSON output and Gradio are part of the required path, not side material.

Strategy 1: Utilize System Prompts
A System Prompt is a special set of instructions given to the model before the user even speaks. It dictates the model's persona and absolute rules. Qwen 2.5 is heavily optimized to obey system prompts.

To ensure formatted output, you must inject this system rule during both training (Step 3) and inference (Step 7).

Here is how you update your inference code (from Step 7) to enforce JSON output using a system role:

In [13]:
#  [Cell 9] [Modified Inference] Enforcing JSON Output
messages = [
    # 1. Add a system prompt with strict formatting rules
    {"role": "system", "content": "You are a helpful assistant. You must ONLY answer in valid JSON format. Do not include any plain text outside the JSON."},

    # 2. Add the user prompt
    {"role": "user", "content": "Who leads the neurology department at MediCore Hospital?"}
]

# Apply the ChatML template (the tokenizer automatically handles the system role)
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)

generated_ids = output[0][inputs.input_ids.shape[1]:]
response_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

print(response_text)

Dr. Elena Salonen leads the neurology department at MediCore Hospital.


In [14]:
# How to update Step 3's preprocess function to include a system prompt:
def preprocess(sample):
    messages = [
        {"role": "system", "content": "You are a helpful assistant. You must ONLY answer in valid JSON format."},
        {"role": "user", "content": sample['prompt']},
        {"role": "assistant", "content": sample['completion']}
    ]
    # ... rest of function unchanged

Model-controlled JSON output (via system prompt)
This version relies entirely on the model following instructions.

A strict system prompt tells the model to output JSON.
If the model was trained well, it will follow the format.
If not, the output may break (invalid JSON, extra text, etc.).
Key idea: You are controlling structure through prompting, not code.
Tradeoff: Simple to implement, but not reliable in production.

In [15]:
#  [Cell 10]
import gradio as gr

# 1. Define the function that Gradio will call when a user submits a prompt
def generate_response(user_prompt):
    messages = [
        # Improved System Prompt: Give the model an exact JSON structure to follow
        {
            "role": "system",
            "content": 'You are a helpful assistant. You must ONLY answer in valid JSON format using the following structure: {"answer": "your detailed response here"}'
        },
        {"role": "user", "content": user_prompt}
    ]

    # Format the text with ChatML tags
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Convert text to tensor numbers and move to GPU
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    # Generate the output
    output = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id
    )

    # Strip out the input prompt
    generated_ids = output[0][inputs.input_ids.shape[1]:]
    response_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return response_text

# 2. Build the Gradio Interface
demo = gr.Interface(
    fn=generate_response,                      # The function to run
    inputs=gr.Textbox(
        lines=3,
        placeholder="e.g. Who leads the neurology department at MediCore Hospital?",
        label="Enter your prompt here"
    ),
    outputs=gr.Textbox(label="Model Output"),  # Where the output will show
    title="MediCore Fine-Tuned Qwen Bot",
    description="Ask questions about MediCore hospital. The model is instructed to reply in JSON format."
)

# 3. Launch the app (share=True creates a public link you can open)
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9381c8521604f56288.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://9381c8521604f56288.gradio.live


For the mini project, this is the safest default path if you want predictable JSON output with the fewest surprises.



In [16]:
#  [Cell 11]
import gradio as gr
import json

def generate_response(user_prompt):
    # Removed the system prompt since the model wasn't trained to use one
    messages = [
        {"role": "user", "content": user_prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id
    )

    generated_ids = output[0][inputs.input_ids.shape[1]:]
    response_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    # --- PYTHON JSON WRAPPER ---
    # We take the raw text and force it into a JSON dictionary
    json_output = json.dumps({"answer": response_text}, indent=4)

    return json_output

demo = gr.Interface(
    fn=generate_response,
    inputs=gr.Textbox(lines=3, label="Enter your prompt here"),
    outputs=gr.Code(language="json", label="JSON Output"), # Changed output to code block
    title="MediCore Fine-Tuned Qwen Bot"
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://16231696f1fdc9b2e3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://16231696f1fdc9b2e3.gradio.live


Pydantic-structured output (best practice)
This is the most robust and scalable method.

A Pydantic schema defines exactly what the output should look like.

The model still generates raw text, but:

It is inserted into a structured object
The structure is validated automatically
Key idea: Treat model output like data that must conform to a schema.

Advantages:

Guaranteed structure
Type validation
Easy to extend (add fields like confidence, sources, etc.)
Best for: APIs, production systems, and real applications



In [17]:
#  [Cell 12]
import gradio as gr
from pydantic import BaseModel, Field

# 1. Define your strict Pydantic Schema
class HospitalResponse(BaseModel):
    # You can add as many fields as you want here
    answer: str = Field(description="The main text answer to the user's question")
    model_version: str = Field(default="Qwen2.5-1.5B-MediCore", description="The model used")

def generate_response(user_prompt):
    messages = [
        {"role": "user", "content": user_prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    # Generate the text
    output = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id
    )

    generated_ids = output[0][inputs.input_ids.shape[1]:]

    # 1. Get the RAW plain text from the model
    raw_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    # 2. Pass the raw text into your Pydantic model
    structured_response = HospitalResponse(answer=raw_text)

    # 3. Use Pydantic to dump it into a perfect JSON string
    json_output = structured_response.model_dump_json(indent=4)

    return json_output

# Build the Gradio Interface
demo = gr.Interface(
    fn=generate_response,
    inputs=gr.Textbox(lines=3, label="Enter your prompt here"),
    outputs=gr.Code(language="json", label="Pydantic JSON Output"),
    title="MediCore Fine-Tuned Qwen Bot (Pydantic Powered)"
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://41cd723e9fd4cea9d4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://41cd723e9fd4cea9d4.gradio.live


Creating gguf file and quantizing it with q4_0

### GGUF Conversion and Quantization to Q4_0

This section will convert your fine-tuned Hugging Face model into the GGUF format and then quantize it to `Q4_0` using `llama.cpp` tools. This allows the model to be run on various devices with improved performance and reduced memory footprint.

In [5]:
import os
import subprocess
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, LoraConfig # Import LoraConfig for explicit config loading
import torch # Ensure torch is imported for device_map and torch_dtype
import sys # Import sys for direct output streaming
import json # Import json for direct config loading

# --- Load the fine-tuned model and tokenizer (from previous steps) ---
# This section ensures 'model' and 'tokenizer' are defined even after a kernel restart.
print("Loading fine-tuned model and tokenizer...")

path = "./my_qwen"

# --- Custom loading of PeftConfig to avoid issues with from_pretrained and local paths ---
adapter_config_path = os.path.join(path, "adapter_config.json")

if not os.path.exists(adapter_config_path):
    print(f"\n--- ERROR: Adapter config file not found ---")
    print(f"Expected to find '{adapter_config_path}', but it does not exist.")
    print("Please ensure you have successfully run the fine-tuning step (Cell 2MOYZH7TME8a) to save the model.")
    print("Exiting GGUF conversion process.")
    sys.exit(1) # Exit if the config is not found

print(f"Loading PEFT configuration from {adapter_config_path}...")
with open(adapter_config_path, 'r') as f:
    config_dict = json.load(f)
# Use LoraConfig directly as we know it's a LoRA adapter
config = LoraConfig(**config_dict)

# Load the tokenizer using the base model name from the config
tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path)
tokenizer.pad_token = tokenizer.eos_token

# Load the base model
base_model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    device_map="cuda",
    torch_dtype=torch.float16 # Use float16 for efficiency
)

# Load the PeftModel (fine-tuned model) by applying adapters to the base model
model = PeftModel.from_pretrained(base_model, path)
model.config.use_cache = True # Re-enable cache for inference
print("Fine-tuned model and tokenizer loaded successfully.")

# --- Step 1: Merge LoRA adapters into the base model ---
print("\nMerging LoRA adapters into the base model...")
merged_model = model.merge_and_unload()

# --- Step 2: Save the merged model ---
merged_model_dir = "my_qwen_merged_hf"
merged_model.save_pretrained(merged_model_dir)
tokenizer.save_pretrained(merged_model_dir)
print(f"Merged Hugging Face model saved to: {merged_model_dir}")

Loading fine-tuned model and tokenizer...
Loading PEFT configuration from ./my_qwen/adapter_config.json...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Fine-tuned model and tokenizer loaded successfully.

Merging LoRA adapters into the base model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged Hugging Face model saved to: my_qwen_merged_hf


In [6]:
# --- Step 3: Setup for GGUF conversion using llama.cpp's convert_hf_to_gguf.py ---
print("\nSetting up llama.cpp for GGUF conversion...")

# Clone llama.cpp if not already present
llama_cpp_dir = "llama.cpp"
if not os.path.exists(llama_cpp_dir):
    print(f"Cloning {llama_cpp_dir} repository...")
    clone_process = subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggerganov/llama.cpp.git", llama_cpp_dir], check=True, capture_output=True, text=True)
    print("--- git clone STDOUT ---")
    print(clone_process.stdout)
    print("--- git clone STDERR ---")
    print(clone_process.stderr)
else:
    print(f"{llama_cpp_dir} already exists, skipping clone.")

# Install dependencies for llama.cpp's convert.py
print("Installing Python dependencies for llama.cpp conversion script...")
!pip install -q sentencepiece numpy protobuf


Setting up llama.cpp for GGUF conversion...
Cloning llama.cpp repository...
--- git clone STDOUT ---

--- git clone STDERR ---
Cloning into 'llama.cpp'...
Updating files:  78% (2162/2738)
Updating files:  79% (2164/2738)
Updating files:  80% (2191/2738)
Updating files:  81% (2218/2738)
Updating files:  82% (2246/2738)
Updating files:  83% (2273/2738)
Updating files:  84% (2300/2738)
Updating files:  85% (2328/2738)
Updating files:  86% (2355/2738)
Updating files:  87% (2383/2738)
Updating files:  88% (2410/2738)
Updating files:  89% (2437/2738)
Updating files:  90% (2465/2738)
Updating files:  91% (2492/2738)
Updating files:  92% (2519/2738)
Updating files:  93% (2547/2738)
Updating files:  94% (2574/2738)
Updating files:  95% (2602/2738)
Updating files:  96% (2629/2738)
Updating files:  97% (2656/2738)
Updating files:  98% (2684/2738)
Updating files:  99% (2711/2738)
Updating files: 100% (2738/2738)
Updating files: 100% (2738/2738), done.

Installing Python dependencies for llama.cpp

In [20]:
# Compile llama.cpp using CMake (if quantize tool doesn't exist)
print("\nEnsuring llama.cpp quantize tool is compiled...")
quantize_tool_path = os.path.join(llama_cpp_dir, "build", "bin", "llama-quantize") # Corrected tool name

if not os.path.exists(quantize_tool_path):
    print("Compiling llama.cpp for quantize tool using CMake...")
    build_dir = os.path.join(llama_cpp_dir, "build")
    os.makedirs(build_dir, exist_ok=True)

    try:
        # Step 1: Run CMake configuration
        print("Running CMake configuration...")
        with open("cmake_configure_stdout.log", "w") as stdout_file, open("cmake_configure_stderr.log", "w") as stderr_file:
            cmake_configure_process = subprocess.run(["cmake", ".."], cwd=build_dir, check=True, capture_output=True, text=True)
            stdout_file.write(cmake_configure_process.stdout)
            stderr_file.write(cmake_configure_process.stderr)
        print("--- CMake Configure STDOUT (from log file) ---")
        with open("cmake_configure_stdout.log", "r") as f:
            sys.stdout.write(f.read())
        print("--- CMake Configure STDERR (from log file) ---")
        with open("cmake_configure_stderr.log", "r") as f:
            sys.stderr.write(f.read())

        # Step 2: Run CMake build (quantize is built by default)
        print("Running CMake build...")
        with open("cmake_build_stdout.log", "w") as stdout_file, open("cmake_build_stderr.log", "w") as stderr_file:
            cmake_build_process = subprocess.run(["cmake", "--build", ".", "--config", "Release"], cwd=build_dir, check=True, capture_output=True, text=True)
            stdout_file.write(cmake_build_process.stdout)
            stderr_file.write(cmake_build_process.stderr)
        print("--- CMake Build STDOUT (from log file) ---")
        with open("cmake_build_stdout.log", "r") as f:
            sys.stdout.write(f.read())
        print("--- CMake Build STDERR (from log file) ---")
        with open("cmake_build_stderr.log", "r") as f:
            sys.stderr.write(f.read())

        # Debugging: List contents of the bin directory and search for quantize
        print(f"\n--- Listing contents of {os.path.dirname(quantize_tool_path)} ---")
        !ls -F {os.path.dirname(quantize_tool_path)}
        print("---------------------------------------------------")
        print(f"\n--- Searching for 'quantize' executable in {build_dir} ---")
        !find {build_dir} -name "llama-quantize*" # Updated find command
        print("---------------------------------------------------")

        if not os.path.exists(quantize_tool_path):
            raise FileNotFoundError(f"quantize executable not found at {quantize_tool_path} after CMake build. See output above for available files and search results.")

    except subprocess.CalledProcessError as e:
        print(f"\n--- ERROR: CMake build failed ---")
        print(f"Command: {e.cmd}")
        print(f"Exit Code: {e.returncode}")
        print(f"STDOUT:\n{e.stdout}")
        print(f"STDERR:\n{e.stderr}")
        print("Compilation of quantize tool failed using CMake. Check cmake_configure_stdout.log, cmake_configure_stderr.log, cmake_build_stdout.log, and cmake_build_stderr.log for full output.")
        sys.exit(1) # Stop execution if this critical step fails
    except FileNotFoundError as e:
        print(f"\n--- ERROR: quantize tool not found after CMake build ---")
        print(e)
        print("Please check the CMake build logs and search results for clues.")
        sys.exit(1)
else:
    print("llama.cpp quantize tool already compiled.")


Ensuring llama.cpp quantize tool is compiled...
llama.cpp quantize tool already compiled.


In [8]:
# --- Step 4: Convert the merged model to GGUF format using llama.cpp's convert_hf_to_gguf.py ---
print("\nConverting merged model to GGUF format using llama.cpp's convert_hf_to_gguf.py (this may take a few minutes)...")

# Define output file names
unquantized_gguf = "my_qwen_merged_f16.gguf" # Unquantized GGUF for intermediate step (using f16)
quantized_gguf = "my_qwen_merged_Q4_0.gguf" # Final quantized GGUF with Q4_0

# Run convert_hf_to_gguf.py
convert_script_path = os.path.join("llama.cpp", "convert_hf_to_gguf.py")
convert_cmd = [
    "python", convert_script_path,
    merged_model_dir,
    "--outtype", "f16", # Generate unquantized GGUF first
    "--outfile", unquantized_gguf
]
print(f"Executing conversion command: {' '.join(convert_cmd)}")

try:
    # Capture output to ensure all messages are seen and written to log files
    with open("convert_stdout.log", "w") as stdout_file, open("convert_stderr.log", "w") as stderr_file:
        convert_process = subprocess.run(convert_cmd, check=True, capture_output=True, text=True)
        stdout_file.write(convert_process.stdout)
        stderr_file.write(convert_process.stderr)

    print("--- convert_hf_to_gguf.py STDOUT (from log file) ---")
    with open("convert_stdout.log", "r") as f:
        sys.stdout.write(f.read())
    print("--- convert_hf_to_gguf.py STDERR (from log file) ---")
    with open("convert_stderr.log", "r") as f:
        sys.stderr.write(f.read())
    print(f"Successfully created unquantized GGUF: {unquantized_gguf}")
except subprocess.CalledProcessError as e:
    print(f"\n--- ERROR: convert_hf_to_gguf.py failed ---")
    print(f"Command: {e.cmd}")
    print(f"Exit Code: {e.returncode}")
    print(f"STDOUT:\n{e.stdout}")
    print(f"STDERR:\n{e.stderr}")
    print("GGUF conversion to F16 failed. Cannot proceed to quantization. Check convert_stdout.log and convert_stderr.log for full output.")
    sys.exit(1) # Stop execution if this critical step fails


Converting merged model to GGUF format using llama.cpp's convert_hf_to_gguf.py (this may take a few minutes)...
Executing conversion command: python llama.cpp/convert_hf_to_gguf.py my_qwen_merged_hf --outtype f16 --outfile my_qwen_merged_f16.gguf
--- convert_hf_to_gguf.py STDOUT (from log file) ---
--- convert_hf_to_gguf.py STDERR (from log file) ---
Successfully created unquantized GGUF: my_qwen_merged_f16.gguf


INFO:hf-to-gguf:Loading model: my_qwen_merged_hf
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> F16, shape = {1536, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float16 --> F16, shape = {8960, 1536}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.float16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.float16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.float16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torch.float16 --> F32, shape = {256}
INFO:hf-to-gguf:blk.0.attn_k.weight,       torch.float16 --> F16, shape = {1536, 256}
INFO:hf-to-gguf:blk.0.attn_output.weight,  torch.float

In [21]:
# --- Step 5: Quantize the GGUF model ---
print("\nQuantizing the GGUF model to Q4_0 format...")

quantize_tool_path = os.path.join("llama.cpp", "build", "bin", "llama-quantize") # Corrected tool name

quantize_cmd = [
    quantize_tool_path,
    unquantized_gguf,
    quantized_gguf,
    "Q4_0" # Quantization method changed to Q4_0 as requested
]
print(f"Executing quantization command: {' '.join(quantize_cmd)}")

try:
    # Capture output to ensure all messages are seen and written to log files
    with open("quantize_stdout.log", "w") as stdout_file, open("quantize_stderr.log", "w") as stderr_file:
        quantize_process = subprocess.run(quantize_cmd, check=True, capture_output=True, text=True)
        stdout_file.write(quantize_process.stdout)
        stderr_file.write(quantize_process.stderr)

    # Display content of log files for user inspection
    print("--- quantize tool STDOUT (from log file) ---")
    with open("quantize_stdout.log", "r") as f:
        sys.stdout.write(f.read())
    print("--- quantize tool STDERR (from log file) ---")
    with open("quantize_stderr.log", "r") as f:
        sys.stderr.write(f.read())
    print(f"Successfully quantized GGUF: {quantized_gguf}")
except subprocess.CalledProcessError as e:
    print(f"\n--- ERROR: quantize tool failed ---")
    print(f"Command: {e.cmd}")
    print(f"Exit Code: {e.returncode}")
    print(f"STDOUT:\n{e.stdout}")
    print(f"STDERR:\n{e.stderr}")
    print("GGUF quantization failed. Check quantize_stdout.log and quantize_stderr.log for full output.")
    sys.exit(1) # Stop execution if this critical step fails


Quantizing the GGUF model to Q4_0 format...
Executing quantization command: llama.cpp/build/bin/llama-quantize my_qwen_merged_f16.gguf my_qwen_merged_Q4_0.gguf Q4_0
--- quantize tool STDOUT (from log file) ---

main: quantize time = 43761.94 ms
main:    total time = 43761.94 ms
--- quantize tool STDERR (from log file) ---
Successfully quantized GGUF: my_qwen_merged_Q4_0.gguf


llama_print_build_info: build = 1 (1ec7ba0)
llama_print_build_info: built with GNU 11.4.0 for Linux x86_64
main: quantizing 'my_qwen_merged_f16.gguf' to 'my_qwen_merged_Q4_0.gguf' as Q4_0
llama_model_loader: loaded meta data with 26 key-value pairs and 338 tensors from my_qwen_merged_f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.800000
llama_model_loader: - kv   4:                      general.sampling.temp f32              = 0.700000
llama_model_loader: - kv   5:                               general.name str            

In [10]:
# --- Step 6: Verify GGUF file creation and provide download instructions ---
print(f"\nVerifying creation of {quantized_gguf}...")
!ls -l {quantized_gguf} || echo "{quantized_gguf} not found by ls."

if os.path.exists(quantized_gguf):
    print(f"\nSuccessfully created GGUF model: {quantized_gguf}")
    print("\n--- GGUF Conversion Complete ---")
    print(f"Your GGUF model '{quantized_gguf}' has been created in the current directory.")
    print("You can download it using the file browser on the left (look for it in /content/).")
    print("Alternatively, you can run the following command in a new cell to download it directly:")
    print(f"```python\nfrom google.colab import files\nfiles.download('{quantized_gguf}')\n```")
else:
    print(f"\n--- ERROR: GGUF Conversion Failed ---\n")
    print(f"The file '{quantized_gguf}' was NOT found after the conversion attempt.")
    print("Please check the output above for any errors or warnings during the conversion or quantization steps.")
    print("Skipping download instructions as GGUF file was not created.")


Verifying creation of my_qwen_merged_Q4_0.gguf...
ls: cannot access 'my_qwen_merged_Q4_0.gguf': No such file or directory
my_qwen_merged_Q4_0.gguf not found by ls.

--- ERROR: GGUF Conversion Failed ---

The file 'my_qwen_merged_Q4_0.gguf' was NOT found after the conversion attempt.
Please check the output above for any errors or warnings during the conversion or quantization steps.
Skipping download instructions as GGUF file was not created.


In [12]:
print('--- Full cmake_build_stderr.log ---')
with open('cmake_build_stderr.log', 'r') as f:
    print(f.read())

--- Full cmake_build_stderr.log ---



In [11]:
print('--- Full cmake_build_stderr.log ---')
with open('cmake_build_stderr.log', 'r') as f:
    print(f.read())

--- Full cmake_build_stderr.log ---



In [13]:
print('--- Full cmake_build_stdout.log ---')
with open('cmake_build_stdout.log', 'r') as f:
    print(f.read())

--- Full cmake_build_stdout.log ---
[  0%] Building C object ggml/src/CMakeFiles/ggml-base.dir/ggml.c.o
[  0%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml.cpp.o
[  1%] Building C object ggml/src/CMakeFiles/ggml-base.dir/ggml-alloc.c.o
[  1%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-backend.cpp.o
[  1%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-backend-meta.cpp.o
[  1%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-opt.cpp.o
[  1%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-threading.cpp.o
[  2%] Building C object ggml/src/CMakeFiles/ggml-base.dir/ggml-quants.c.o
[  2%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/gguf.cpp.o
[  2%] Linking CXX shared library ../../bin/libggml-base.so
[  2%] Built target ggml-base
[  2%] Building C object ggml/src/CMakeFiles/ggml-cpu.dir/ggml-cpu/ggml-cpu.c.o
[  2%] Building CXX object ggml/src/CMakeFiles/ggml-cpu.dir/ggml-cpu/ggml-cpu.cpp.o
[  3%] Building CXX object